# Dynamics analysis for the `sameloc` model

Trajectory **tangling**, **participation ratio (PR)**, and the **dPCA variance split**
(condition-independent `t` / pure-CTOA offset `s` / interaction `ts`) for the sameloc
topo model — the variant where cue and target appear at the SAME random 2D location
(`CuedTargetSameLoc2D`), trained to `stage1_topo_sameloc.pt`.

Mirrors the topo-model analyses in `04b_tangling_topo.ipynb`,
`12_stage2_uniform_test.ipynb`, and `02b_analysis_topo.ipynb`, reusing the same
`src/analysis.py` / `src/plotting.py` functions so the sameloc variant can be compared
against the main topo model and Amengual's monkey reference.

In [ ]:
import sys, os, pathlib
ROOT = next(p for p in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]
            if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

import numpy as np
import torch
import matplotlib.pyplot as plt
from collections import defaultdict
from sklearn.decomposition import PCA

from src.model_topo import BioLeakyRNNTopo
from src.env import CuedTargetSameLoc2D
from src.analysis import (
    collect_trials_spatial,
    filter_trials,
    tangling_by_ctoa_bin,
    pr_singletrial_by_ctoa_bin,
    participation_ratio,
    polynomial_regression,
    collect_aligned_hidden_by_label,
    make_condition_mean_tensor,
)
from src.plotting import plot_tangling_timecourses, plot_tangling_vs_ctoa

device = "cpu"


def make_model():
    return BioLeakyRNNTopo(
        input_size=7, hidden_size=180, output_size=2, dt=20.0, tau=100.0,
        activation="softplus", sigma_rec=0.10, rec_init="diag", use_ei=True,
        exc_ratio=0.80, use_dale=True, mask_seed=42, sheet_side=12,
        tau_ee=0.25, tau_ie=0.32, tau_ei=0.64, tau_ii=0.64, rf_sigma=0.3,
        tau_e_range=(50, 150), tau_i_range=(10, 50), use_adaptation=False,
    ).to(device)


model = make_model()
ckpt = torch.load("checkpoints/stage1_topo_sameloc.pt", weights_only=True)
model.load_state_dict(ckpt["state_dict"], strict=False)
model.eval()
model.noise_at_eval = True


def make_env():
    return CuedTargetSameLoc2D(dt=20, cue_strength=1.0,
                               p_distractor_trial=0.0, distractor_strength=0.0)


# 2500 trials (vs 800 in 01g) so each of the 10 CTOA bins clears min_trials for
# per-bin tangling / PR estimates.
torch.manual_seed(0)
np.random.seed(0)
trials = collect_trials_spatial(model, make_env, n_trials=2500,
                                device=device, head="go_nogo")
corr = filter_trials(trials, outcome="correct")
print(f"correct: {len(corr)} / {len(trials)}")

bin_counts = defaultdict(int)
for tr in corr:
    bin_counts[tr["ctoa_bin"]] += 1
print("Correct-trial CTOA bin counts:", dict(sorted(bin_counts.items())))

## 1. Trajectory tangling

Russo et al. (2018) tangling `Q`, computed per CTOA bin on PCA-reduced
condition-averaged trajectories, in a pre-target (300 ms before target onset) and a
post-target (200 ms after) window.

In [ ]:
tang_pre = tangling_by_ctoa_bin(
    corr, align_key="target_onset",
    window_before=15, window_after=0,   # 300 ms pre-target
    pca_dims=20, outcome="correct", min_trials=10,
)
tang_post = tangling_by_ctoa_bin(
    corr, align_key="target_onset",
    window_before=0, window_after=10,   # 200 ms post-target
    pca_dims=20, outcome="correct", min_trials=10,
)

print("Pre-target  Q_mean per bin:", np.round(tang_pre["Q_mean"], 4))
print("Post-target Q_mean per bin:", np.round(tang_post["Q_mean"], 4))

plot_tangling_timecourses(tang_pre, tang_post)
plot_tangling_vs_ctoa(tang_pre, tang_post)

## 2. Participation ratio (dimensionality)

Single-trial PR per CTOA bin on the top-3 shared PCs (pre-target window), mirroring
`12_stage2_uniform_test.ipynb`. Paper reports an inverted-U (PR peaks at intermediate
CTOA).

In [ ]:
def ctoa_ms_mean(trials, labels, outcome="correct"):
    bin_to_ms = defaultdict(list)
    for tr in trials:
        if tr.get("train_outcome") != outcome:
            continue
        b, ms = tr.get("ctoa_bin"), tr.get("ctoa_ms")
        if b in labels and ms is not None:
            bin_to_ms[b].append(ms)
    return np.array([np.mean(bin_to_ms[b]) if b in bin_to_ms else np.nan
                      for b in labels])


def quad_fit(x, y):
    x = np.asarray(x, dtype=float); y = np.asarray(y, dtype=float)
    m = np.isfinite(x) & np.isfinite(y)
    if m.sum() < 3:
        return None
    p = np.polyfit(x[m], y[m], 2)
    yhat = np.polyval(p, x[m])
    ss_res = np.sum((y[m] - yhat) ** 2)
    ss_tot = np.sum((y[m] - y[m].mean()) ** 2)
    R2 = 1 - ss_res / ss_tot if ss_tot > 0 else float("nan")
    xs = np.linspace(x[m].min(), x[m].max(), 200)
    return p, xs, np.polyval(p, xs), R2


pr3 = pr_singletrial_by_ctoa_bin(
    corr, window_before=15, window_after=0, n_components=3,
    outcome="correct", min_trials=10,
)
pr3["ctoa_ms_mean"] = ctoa_ms_mean(corr, pr3["labels"])

print(f"PR-3 explained variance: {pr3['explained']:.3f}")
print(f"PR per bin: {np.round(pr3['PR'], 4)}")
pk = int(np.argmax(pr3["PR"]))
print(f"Peak bin:   {pk}  (CTOA={pr3['ctoa_ms_mean'][pk]:.0f} ms)")

fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(pr3["ctoa_ms_mean"], pr3["PR"], s=55, color="#ed7d31",
           edgecolor="k", linewidth=0.5, zorder=3)
fit = quad_fit(pr3["ctoa_ms_mean"], pr3["PR"])
if fit is not None:
    p, xs, ys, R2 = fit
    ax.plot(xs, ys, "--", color="red", lw=1.5, zorder=2)
    ax.text(0.62, 0.85, f"R^2 = {R2:.2f}", transform=ax.transAxes,
            color="red", fontsize=11)
    print(f"Quadratic fit: a={p[0]:+.3e}  (inverted-U iff a<0)")
ax.set_xlabel("CTOA (ms)"); ax.set_ylabel("Dimensionality (PR on 3 PCs)")
ax.set_title("sameloc: PR-3 vs CTOA")
ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 3. dPCA variance split (monkey vs RNN)

Orthogonal marginalization of the condition-averaged activity (window
`[-400, +400]` ms) into condition-independent `t`, pure-CTOA offset `s`, and
interaction `ts`. Donut layout reproduced from `02b_analysis_topo.ipynb`; the Monkey
panel `[79, 21]` is the Amengual Fig 5D reference.

In [ ]:
from matplotlib.patches import Patch


def _marginal_split(window_before, window_after):
    by, _ = collect_aligned_hidden_by_label(
        corr, label_fn=lambda tr: tr.get("ctoa_bin"),
        align_key="target_onset", window_before=window_before, window_after=window_after)
    X, _, _ = make_condition_mean_tensor(by, min_trials=5)        # [C, T, H]
    mu = X.mean((0, 1), keepdims=True)
    phi_t = X.mean(0, keepdims=True) - mu       # time only (condition-independent)
    phi_c = X.mean(1, keepdims=True) - mu       # pure-CTOA offset s
    phi_ct = X - mu - phi_t - phi_c             # interaction ts
    ss = lambda a: float((np.broadcast_to(a, X.shape) ** 2).sum())
    t, s, ts = ss(phi_t), ss(phi_c), ss(phi_ct)
    tot = t + s + ts
    return t / tot, s / tot, ts / tot


t20, s20, ts20 = _marginal_split(20, 20)   # paper-matched [-400, +400] ms
print(f"[-400,+400] ms  CI(time)={t20:.1%}  pure-CTOA s={s20:.1%}  interaction ts={ts20:.1%}")
print(f"   2-way (paper style): CI={t20:.1%}   CTOA-related={s20+ts20:.1%}")
print(f"   Amengual Fig 5D:     CI=79.0%   CTOA-related=21.0%")

GREY, GOLD, ORANGE = "#bdbdbd", "#f0c05a", "#e8845b"
pct = dict(autopct="%1.0f%%", pctdistance=0.80, startangle=90, counterclock=False,
           wedgeprops=dict(width=0.45, edgecolor="w"),
           textprops=dict(color="white", fontsize=11, fontweight="bold"))
fig, ax = plt.subplots(1, 3, figsize=(12, 5.0))
ax[0].pie([79, 21], colors=[GREY, ORANGE], **pct)
ax[0].set_title("Monkey" + chr(10) + "[-400,+400] ms")
ax[1].pie([t20 * 100, (s20 + ts20) * 100], colors=[GREY, ORANGE], **pct)
ax[1].set_title("RNN sameloc" + chr(10) + "(same window)")
ax[2].pie([t20 * 100, s20 * 100, ts20 * 100], colors=[GREY, GOLD, ORANGE], **pct)
ax[2].set_title("RNN sameloc: CTOA split" + chr(10) + "into s vs ts")
fig.legend(handles=[
    Patch(color=GREY, label="condition-independent (t)"),
    Patch(color=GOLD, label="pure-CTOA offset (s)"),
    Patch(color=ORANGE, label="CTOA-related / interaction (ts)")],
    loc="lower center", ncol=3, frameon=False, fontsize=10)
fig.suptitle("dPCA variance split: monkey vs RNN sameloc model", fontsize=13, y=0.99)
fig.subplots_adjust(top=0.80, bottom=0.10, wspace=0.1)
fig.savefig("figures/dpca_variance_split_sameloc.png", dpi=140)
plt.show()

### Post-target-only split (fairer monkey comparison)

The `[-400,+400]` window above includes the pre-target part, where the per-CTOA
cue-timing shift leaks into `s` and inflates the CTOA-related fraction. Restricting to
the post-target window (`window_before=0`) removes that artifact, so this split reflects
genuine post-target CTOA coding — the cleaner number to compare against the monkey.

In [ ]:
# Reuses _marginal_split, GREY/GOLD/ORANGE, pct from the cell above.
tp, sp, tsp = _marginal_split(0, 20)   # [0, +400] ms, post-target only
print(f"[0,+400] ms post-target  CI(time)={tp:.1%}  pure-CTOA s={sp:.1%}  interaction ts={tsp:.1%}")
print(f"   2-way (paper style): CI={tp:.1%}   CTOA-related={sp + tsp:.1%}")
print(f"   Amengual Fig 5D:     CI=79.0%   CTOA-related=21.0%")

fig, ax = plt.subplots(1, 3, figsize=(12, 5.0))
ax[0].pie([79, 21], colors=[GREY, ORANGE], **pct)
ax[0].set_title("Monkey" + chr(10) + "[-400,+400] ms")
ax[1].pie([tp * 100, (sp + tsp) * 100], colors=[GREY, ORANGE], **pct)
ax[1].set_title("RNN sameloc" + chr(10) + "post-target [0,+400] ms")
ax[2].pie([tp * 100, sp * 100, tsp * 100], colors=[GREY, GOLD, ORANGE], **pct)
ax[2].set_title("RNN sameloc post-target:" + chr(10) + "s vs ts")
fig.legend(handles=[
    Patch(color=GREY, label="condition-independent (t)"),
    Patch(color=GOLD, label="pure-CTOA offset (s)"),
    Patch(color=ORANGE, label="CTOA-related / interaction (ts)")],
    loc="lower center", ncol=3, frameon=False, fontsize=10)
fig.suptitle("dPCA variance split (post-target only): monkey vs RNN sameloc", fontsize=13, y=0.99)
fig.subplots_adjust(top=0.80, bottom=0.10, wspace=0.1)
fig.savefig("figures/dpca_variance_split_sameloc_posttarget.png", dpi=140)
plt.show()

## Interpretation

Fill in after running, comparing sameloc against the main topo model and Amengual:

- **Tangling:** does pre/post-target `Q` modulate with CTOA (inverted-U), or stay flat?
- **PR:** is the PR-3 vs CTOA quadratic inverted-U (`a < 0`, like the paper), or flat?
- **dPCA split:** how does the sameloc condition-independent vs CTOA-related fraction
  compare to the monkey 79/21 and to the main topo model?